In [47]:
# ============================================================
# CELL 1 — Check Python Environment
# ============================================================

import sys

print("Python version:")
print(sys.version)

print("\nPython executable:")
print(sys.executable)

Python version:
3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]

Python executable:
/usr/bin/python3


In [48]:
import sys

print("Python version:", sys.version)

Python version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]


In [50]:
!pip install -q -U "crewai[litellm]" litellm groq python-dotenv pydantic

In [1]:
import os
import json

from getpass import getpass

from crewai import Agent, Task, Crew, Process, LLM
from pydantic import BaseModel, Field

In [2]:
# ============================================================
# CREWAI + GROQ COMPATIBILITY FIX
# ============================================================

import crewai.llms.cache as crew_cache

# CrewAI may add cache_breakpoint=True to messages.
# Groq does not accept this property.
crew_cache.mark_cache_breakpoint = lambda msg: msg

print("✅ Groq cache_breakpoint workaround applied.")

✅ Groq cache_breakpoint workaround applied.


In [3]:
# ============================================================
# GROQ API KEY
# ============================================================

if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass(
        "Enter your Groq API key: "
    )

print("✅ GROQ_API_KEY loaded.")

Enter your Groq API key: ··········
✅ GROQ_API_KEY loaded.


In [4]:
# ============================================================
# CELL 6 — GROQ LLM CONFIGURATION
# ============================================================

MODEL_NAME = "groq/openai/gpt-oss-20b"

llm = LLM(
    model=MODEL_NAME,
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0.1,
    max_tokens=1800
)

print("✅ LLM configured")
print("Model:", MODEL_NAME)
print("Max output tokens: 1800")

✅ LLM configured
Model: groq/openai/gpt-oss-20b
Max output tokens: 1800


In [5]:
# ============================================================
# CELL 7 — SMALL LLM TEST
# ============================================================

test_agent = Agent(
    role="Test Assistant",
    goal="Give a very short confirmation.",
    backstory="You are a testing assistant.",
    llm=llm,
    verbose=False,
    allow_delegation=False
)

test_task = Task(
    description="Reply with exactly: Leo is ready.",
    expected_output="Leo is ready.",
    agent=test_agent
)

test_crew = Crew(
    agents=[test_agent],
    tasks=[test_task],
    process=Process.sequential,
    verbose=False
)

try:

    test_result = await test_crew.kickoff_async()

    print("✅ LLM TEST SUCCESSFUL")
    print(test_result.raw)

except Exception as e:

    print("❌ LLM TEST FAILED")
    print(type(e).__name__)
    print(str(e))

✅ LLM TEST SUCCESSFUL
Leo is ready.


In [6]:
# ============================================================
# CELL 8 — QUIZ SCHEMA
# ============================================================

from typing import Literal
from pydantic import BaseModel, Field


class QuizQuestion(BaseModel):

    question: str

    options: list[str]

    correct_answer: Literal["A", "B", "C", "D"]

    explanation: str


class Quiz(BaseModel):

    topic: str

    questions: list[QuizQuestion]


print("✅ Quiz schema created")

✅ Quiz schema created


In [7]:
# ============================================================
# CELL 9 — LEO MEMORY
# ============================================================

leo_memory = {
    "student_name": None,
    "topic": None,
    "level": None,
    "language": None,
    "quiz_score": None
}

print("✅ Leo memory initialized")

✅ Leo memory initialized


In [8]:
# ============================================================
# CELL 10 — AGENT PROMPT TEMPLATES
# ============================================================

COORDINATOR_PROMPT = """
You are Leo's Coordinator.

Your responsibility is to understand the student's learning request
and organize the learning workflow.

Student:
- Name: {student_name}
- Topic: {topic}
- Level: {level}
- Language: {language}

The workflow must contain:

1. An explanation from the Explainer.
2. A quiz from the Quiz Master.
3. Evaluation from the Evaluator after the student answers.

Do not teach the topic yourself.
Do not create the quiz yourself.
Focus on understanding the request and organizing the workflow.
"""


EXPLAINER_PROMPT = """
You are Leo's Explainer.

Your job is to teach the requested topic clearly.

Student:
- Name: {student_name}
- Topic: {topic}
- Level: {level}
- Language: {language}

Requirements:

1. Explain the topic according to the student's level.
2. Use simple and understandable language.
3. Give useful examples.
4. Organize the lesson with headings.
5. Focus only on the requested topic.
6. Prepare the student for a quiz.

If the language is Bangla, explain primarily in Bangla.
"""


QUIZ_MASTER_PROMPT = """
You are Leo's Quiz Master.

Create a multiple-choice quiz based ONLY on the lesson provided
by the Explainer.

Student:
- Name: {student_name}
- Topic: {topic}
- Level: {level}
- Language: {language}

Requirements:

1. Create exactly 5 questions.
2. Each question must have exactly 4 options.
3. Options must start with:
   A)
   B)
   C)
   D)

4. correct_answer must contain only:
   A, B, C, or D

5. Questions must be based on the lesson.
6. Mix easy, medium, and challenging questions.
7. Include a short explanation for every correct answer.
8. Use the requested language.
9. Do not create questions unrelated to the lesson.
"""

In [9]:
# ============================================================
# CELL 11 — COORDINATOR AGENT
# ============================================================

coordinator = Agent(
    role="Coordinator",
    goal=(
        "Understand the student's request and coordinate "
        "the complete learning workflow."
    ),
    backstory=COORDINATOR_PROMPT,
    llm=llm,
    verbose=True,
    allow_delegation=False
)

print("✅ Coordinator agent created")

✅ Coordinator agent created


In [10]:
# ============================================================
# CELL 12 — EXPLAINER AGENT
# ============================================================

explainer = Agent(
    role="Explainer",
    goal=(
        "Teach the requested topic clearly according "
        "to the student's level."
    ),
    backstory=EXPLAINER_PROMPT,
    llm=llm,
    verbose=True,
    allow_delegation=False
)

print("✅ Explainer agent created")

✅ Explainer agent created


In [11]:
# ============================================================
# CELL 13 — QUIZ MASTER AGENT
# ============================================================

quiz_master = Agent(
    role="Quiz Master",
    goal=(
        "Create a structured five-question quiz based "
        "on the Explainer's lesson."
    ),
    backstory=QUIZ_MASTER_PROMPT,
    llm=llm,
    verbose=True,
    allow_delegation=False
)

print("✅ Quiz Master agent created")

✅ Quiz Master agent created


In [12]:
# ============================================================
# CELL 14 — EVALUATOR AGENT
# ============================================================

EVALUATOR_PROMPT = """
You are Leo's Evaluator.

Your job is to evaluate the student's quiz answers.

You will receive:

1. The original structured quiz.
2. The student's submitted answers.

Requirements:

1. Check every answer against the provided correct answer.
2. Identify correct answers.
3. Identify incorrect answers.
4. Explain mistakes.
5. Give constructive feedback.
6. Recommend what the student should study next.
7. Do not invent questions.
8. Do not change the correct answers.
9. Use the requested language.
"""

evaluator = Agent(
    role="Evaluator",
    goal=(
        "Evaluate the student's quiz answers accurately "
        "and provide useful learning feedback."
    ),
    backstory=EVALUATOR_PROMPT,
    llm=llm,
    verbose=True,
    allow_delegation=False
)

print("✅ Evaluator agent created")

✅ Evaluator agent created


In [13]:
# ============================================================
# CELL 15 — STUDENT INPUT
# ============================================================

print("=" * 60)
print("🎓 LEO — AI TUTOR")
print("=" * 60)

student_name = input("Enter your name: ").strip()
topic = input("What topic do you want to learn? ").strip()
level = input("What is your level? (Beginner/Intermediate/Advanced): ").strip()
language = input("Preferred language? (English/Bangla): ").strip()

# Basic validation
if not student_name:
    raise ValueError("Student name cannot be empty.")

if not topic:
    raise ValueError("Topic cannot be empty.")

if not level:
    raise ValueError("Level cannot be empty.")

if not language:
    raise ValueError("Language cannot be empty.")

# Store student information in Leo's memory
leo_memory["student_name"] = student_name
leo_memory["topic"] = topic
leo_memory["level"] = level
leo_memory["language"] = language

print("\n" + "=" * 60)
print("✅ STUDENT INFORMATION SAVED")
print("=" * 60)

print(f"Student : {student_name}")
print(f"Topic   : {topic}")
print(f"Level   : {level}")
print(f"Language: {language}")

🎓 LEO — AI TUTOR
Enter your name: Sneha
What topic do you want to learn? Literature
What is your level? (Beginner/Intermediate/Advanced): Intermediate
Preferred language? (English/Bangla): Bangla

✅ STUDENT INFORMATION SAVED
Student : Sneha
Topic   : Literature
Level   : Intermediate
Language: Bangla


In [14]:
# ============================================================
# CELL 16 — COORDINATOR TASK
# ============================================================

coordinator_task = Task(
    description=f"""
    Student name: {student_name}
    Topic: {topic}
    Level: {level}
    Language: {language}

    Analyze this learning request and create a clear
    learning workflow for Leo.

    The workflow must include:

    1. A lesson from the Explainer.
    2. A quiz based on that lesson.
    3. Evaluation after the student submits answers.

    The Coordinator should focus on delegation and workflow
    management rather than performing the specialist tasks.

    Do not explain the topic.
    Do not create quiz questions.
    """,

    expected_output="""
    A concise learning workflow explaining:

    - What the Explainer should do
    - What the Quiz Master should do
    - What the student should do
    - What the Evaluator should do
    """,

    agent=coordinator
)

print("✅ Coordinator task created")

✅ Coordinator task created


In [15]:
# ============================================================
# CELL 17 — EXPLAINER TASK
# ============================================================

explainer_task = Task(
    description=f"""
    Student name: {student_name}
    Topic: {topic}
    Level: {level}
    Language: {language}

    Teach the student the requested topic.

    Follow the learning workflow prepared by the Coordinator.

    Explain:
    - The main concept
    - Important points
    - Relevant examples
    - Key ideas the student should remember

    The lesson must prepare the student for the quiz.
    """,

    expected_output="""
    A complete but understandable lesson on the requested topic,
    appropriate for the student's level and language.
    """,

    agent=explainer,

    context=[coordinator_task]
)

print("✅ Explainer task created")

✅ Explainer task created


In [16]:
# ============================================================
# CELL 18 — QUIZ MASTER TASK
# ============================================================

quiz_task = Task(
    description=f"""
Student: {student_name}
Topic: {topic}
Level: {level}
Language: {language}

Read the lesson produced by the Explainer.

Create exactly 5 multiple-choice questions.

Each question must contain:
- question
- options
- correct_answer
- explanation

Each question must have exactly four options:
A) ...
B) ...
C) ...
D) ...

correct_answer must be exactly:
A, B, C, or D

Return ONLY valid JSON.

Use exactly this structure:

{{
  "topic": "{topic}",
  "questions": [
    {{
      "question": "Question text",
      "options": [
        "A) Option",
        "B) Option",
        "C) Option",
        "D) Option"
      ],
      "correct_answer": "A",
      "explanation": "Explanation"
    }}
  ]
}}

Do not write Markdown.
Do not write ```json.
Do not add text before or after the JSON.
""",

    expected_output="""
    A valid JSON object containing exactly 5 multiple-choice
    questions based on the Explainer's lesson.
    """,

    agent=quiz_master,

    context=[explainer_task]
)

print("✅ Quiz Master task created")

✅ Quiz Master task created


In [17]:
# ============================================================
# CELL 19 — TEACHING CREW
# ============================================================

teaching_crew = Crew(
    agents=[
        coordinator,
        explainer,
        quiz_master
    ],

    tasks=[
        coordinator_task,
        explainer_task,
        quiz_task
    ],

    process=Process.sequential,

    verbose=True
)

print("✅ Teaching crew created")

✅ Teaching crew created


In [18]:
# ============================================================
# CELL 20 — RUN TEACHING WORKFLOW
# ============================================================

import asyncio
import time


teaching_inputs = {
    "student_name": student_name,
    "topic": topic,
    "level": level,
    "language": language
}


print("=" * 80)
print("LEO — PHASE 1")
print("COORDINATOR → EXPLAINER → QUIZ MASTER")
print("=" * 80)


try:

    teaching_result = await teaching_crew.kickoff_async(
        inputs=teaching_inputs
    )

    print("\n")
    print("=" * 80)
    print("✅ TEACHING PHASE COMPLETED")
    print("=" * 80)

except Exception as e:

    print("\n")
    print("=" * 80)
    print("❌ LEO ERROR")
    print("=" * 80)

    print("Error type:", type(e).__name__)
    print("Error:", str(e))

LEO — PHASE 1
COORDINATOR → EXPLAINER → QUIZ MASTER


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: a1c45ede-b30a-4797-8793-f3e401f981fd                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│      Student name: Sneha                                                                                        │
│      Topic: Literature                                                                                          │
│      Level: Intermediate                                                                                        │
│      Language: Bangla                                                                                           │
│                                                                                                                 │
│      Analyze this learning request and create a clear                                                           │
│      learning workflow for Leo.                                                                                 │
│                                                                                                                 │
│      The workflow must include:                                                                                 │
│                                                                                                                 │
│      1. A lesson from the Explainer.                                                                            │
│      2. A quiz based on that lesson.                                                                            │
│      3. Evaluation after the student submits answers.                                                           │
│                                                                                                                 │
│      The Coordinator should focus on delegation and workflow                                                    │
│      management rather than performing the specialist tasks.                                                    │
│                                                                                                                 │
│      Do not explain the topic.                                                                                  │
│      Do not create quiz questions.                                                                              │
│                                                                                                                 │
│  ID: 40db70a8-9095-428a-a231-672dfe142793                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Coordinator                                                                                             │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Student name: Sneha                                                                                        │
│      Topic: Literature                                                                                          │
│      Level: Intermediate                                                                                        │
│      Language: Bangla                                                                                           │
│                                                                                                                 │
│      Analyze this learning request and create a clear                                                           │
│      learning workflow for Leo.                                                                                 │
│                                                                                                                 │
│      The workflow must include:                                                                                 │
│                                                                                                                 │
│      1. A lesson from the Explainer.                                                                            │
│      2. A quiz based on that lesson.                                                                            │
│      3. Evaluation after the student submits answers.                                                           │
│                                                                                                                 │
│      The Coordinator should focus on delegation and workflow                                                    │
│      management rather than performing the specialist tasks.                                                    │
│                                                                                                                 │
│      Do not explain the topic.                                                                                  │
│      Do not create quiz questions.                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Quiz Master                                                                                             │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Student: Sneha                                                                                                 │
│  Topic: Literature                                                                                              │
│  Level: Intermediate                                                                                            │
│  Language: Bangla                                                                                               │
│                                                                                                                 │
│  Read the lesson produced by the Explainer.                                                                     │
│                                                                                                                 │
│  Create exactly 5 multiple-choice questions.                                                                    │
│                                                                                                                 │
│  Each question must contain:                                                                                    │
│  - question                                                                                                     │
│  - options                                                                                                      │
│  - correct_answer                                                                                               │
│  - explanation                                                                                                  │
│                                                                                                                 │
│  Each question must have exactly four options:                                                                  │
│  A) ...                                                                                                         │
│  B) ...                                                                                                         │
│  C) ...                                                                                                         │
│  D) ...                                                                                                         │
│                                                                                                                 │
│  correct_answer must be exactly:                                                                                │
│  A, B, C, or D                                                                                                  │
│                                                                                                                 │
│  Return ONLY valid JSON.                                                                                        │
│                                                                                                                 │
│  Use exactly this structure:                                                                                    │
│                                                                                                                 │
│  {                                                                                                              │
│    "topic": "Literature",                              

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Quiz Master                                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "topic": "Literature",                                                                                       │
│    "questions": [                                                                                               │
│      {                                                                                                          │
│        "question": "গল্পের থিম শনাক্ত করার প্রথম ধাপ কী?",                                                               │
│        "options": [                                                                                             │
│          "A) চরিত্রের বর্ণনা পড়া",                                                                                    │
│          "B) প্রশ্ন করা",                                                                                          │
│          "C) শব্দ গণনা করা",                                                                                       │
│          "D) ছবি আঁকা"                                                                                             │
│        ],                                                                                                       │
│        "correct_answer": "B",                                                                                   │
│        "explanation": "প্রথম ধাপ হলো গল্পটি মনোযোগ দিয়ে পড়া এবং প্রশ্ন করা, যাতে থিমের ইঙ্গিত পাওয়া যায়।"                              │
│      },                                                                                                         │
│      {                                                                                                          │
│        "question": "থিমকে শক্তিশালী করার উপায় নয় কোনটি?",                                                                 │
│        "options": [                                                                                             │
│          "A) প্রতীক ব্যবহার",                                                                                       │
│          "B) পুনরাবৃত্তি",                                                                                           │
│          "C) চরিত্রের নাম পরিবর্তন",                                                                                  │
│          "D) রূপক"                                                                                               │
│        ],                                                                                                       │
│        "correct_answer": "C",                                                                                   │
│        "explanation": "চরিত্রের নাম পরিবর্তন থিমকে শক্তিশালী করে না; প্রতীক, পুনরাবৃত্তি ও রূপকই থিমকে জোরদার করে।"                        │
│      },                                                                                                         │
│      {                                                                                                          │
│        "question": "\"মেঘের ছায়া\" গল্পে কোন প্রতীক ব্যবহার করা হয়েছে এবং তা কী প্রকাশ করে?",                                       │
│        "options": [                                                                                             │
│          "A) পাখি – স্বাধীনতা",           

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Student: Sneha                                                                                                 │
│  Topic: Literature                                                                                              │
│  Level: Intermediate                                                                                            │
│  Language: Bangla                                                                                               │
│                                                                                                                 │
│  Read the lesson produced by the Explainer.                                                                     │
│                                                                                                                 │
│  Create exactly 5 multiple-choice questions.                                                                    │
│                                                                                                                 │
│  Each question must contain:                                                                                    │
│  - question                                                                                                     │
│  - options                                                                                                      │
│  - correct_answer                                                                                               │
│  - explanation                                                                                                  │
│                                                                                                                 │
│  Each question must have exactly four options:                                                                  │
│  A) ...                                                                                                         │
│  B) ...                                                                                                         │
│  C) ...                                                                                                         │
│  D) ...                                                                                                         │
│                                                                                                                 │
│  correct_answer must be exactly:                                                                                │
│  A, B, C, or D                                                                                                  │
│                                                                                                                 │
│  Return ONLY valid JSON.                                                                                        │
│                                                                                                                 │
│  Use exactly this structure:                                                                                    │
│                                                                                                                 │
│  {                                                                                                              │
│    "topic": "Literature",                                                                                       │
│    "questions": [                                      



✅ TEACHING PHASE COMPLETED


In [19]:
# ============================================================
# CELL 21 — GET LESSON
# ============================================================

print("=" * 80)
print("📚 LESSON")
print("=" * 80)

explainer_output = explainer_task.output

lesson_text = explainer_output.raw

print(lesson_text)

📚 LESSON
## থিম্যাটিক বিশ্লেষণ: একটি মধ্যম স্তরের সাহিত্য পাঠ  

### ১. ভূমিকা  
- **থিম** হলো গল্পের মূল ভাবনা বা বার্তা।  
- এটি লেখকের উদ্দেশ্য প্রকাশ করে এবং পাঠকের অনুভূতি ও চিন্তাকে প্রভাবিত করে।  
- মধ্যম স্তরের শিক্ষার্থীরা থিম শনাক্ত করে গল্পের গভীরতা বুঝতে পারে।  

### ২. থিম কী?  
| শব্দ | ব্যাখ্যা |
|------|----------|
| **থিম** | গল্পের কেন্দ্রীয় ভাবনা, যেমন “প্রেমের মূল্য”, “স্বাধীনতার সংগ্রাম” ইত্যাদি। |
| **সাব-থিম** | প্রধান থিমকে সমর্থনকারী ছোট ভাবনা, যেমন “বন্ধুত্বের শক্তি”। |
| **প্রতীক** | থিমকে প্রকাশ করার জন্য ব্যবহৃত বস্তু বা ঘটনা, যেমন “পাখি” = স্বাধীনতা। |

### ৩. থিম শনাক্ত করার ধাপ  
1. **গল্পটি মনোযোগ দিয়ে পড়ুন** – চরিত্র, ঘটনা, পরিবেশ সবই লক্ষ্য করুন।  
2. **প্রশ্ন করুন** – “গল্পের মূল বার্তা কী?” “লেখক কী বলতে চেয়েছেন?”  
3. **পুনরাবৃত্তি খুঁজুন** – একই রকম ঘটনা বা বাক্যাংশ বারবার আসলে তা থিমের ইঙ্গিত।  
4. **প্রতীক ও রূপক চিহ্নিত করুন** – এগুলো থিমকে শক্তিশালী করে।  
5. **উপসংহার টানুন** – আপনার পর্যবেক্ষণগুলোকে একটি বাক্যে সংক্ষেপ করুন।  

### ৪. উদ

In [20]:
# ============================================================
# CELL 22 — GET RAW QUIZ
# ============================================================

quiz_output = quiz_task.output

quiz_text = quiz_output.raw

print("=" * 80)
print("📝 RAW QUIZ OUTPUT")
print("=" * 80)

print(quiz_text)

📝 RAW QUIZ OUTPUT
{
  "topic": "Literature",
  "questions": [
    {
      "question": "গল্পের থিম শনাক্ত করার প্রথম ধাপ কী?",
      "options": [
        "A) চরিত্রের বর্ণনা পড়া",
        "B) প্রশ্ন করা",
        "C) শব্দ গণনা করা",
        "D) ছবি আঁকা"
      ],
      "correct_answer": "B",
      "explanation": "প্রথম ধাপ হলো গল্পটি মনোযোগ দিয়ে পড়া এবং প্রশ্ন করা, যাতে থিমের ইঙ্গিত পাওয়া যায়।"
    },
    {
      "question": "থিমকে শক্তিশালী করার উপায় নয় কোনটি?",
      "options": [
        "A) প্রতীক ব্যবহার",
        "B) পুনরাবৃত্তি",
        "C) চরিত্রের নাম পরিবর্তন",
        "D) রূপক"
      ],
      "correct_answer": "C",
      "explanation": "চরিত্রের নাম পরিবর্তন থিমকে শক্তিশালী করে না; প্রতীক, পুনরাবৃত্তি ও রূপকই থিমকে জোরদার করে।"
    },
    {
      "question": "\"মেঘের ছায়া\" গল্পে কোন প্রতীক ব্যবহার করা হয়েছে এবং তা কী প্রকাশ করে?",
      "options": [
        "A) পাখি – স্বাধীনতা",
        "B) মেঘ – অস্থায়ী বাধা",
        "C) সূর্য – আশা",
        "D) নদী – প্রবাহ"
 

In [21]:
# ============================================================
# CELL 23 — CLEAN QUIZ JSON
# ============================================================

import json
import re


def clean_json_text(text):

    text = text.strip()

    # Remove ```json
    text = re.sub(
        r"^```json\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

    # Remove ```
    text = re.sub(
        r"\s*```$",
        "",
        text
    )

    return text.strip()


cleaned_quiz_text = clean_json_text(
    quiz_text
)

print("Cleaned quiz:")
print(cleaned_quiz_text)

Cleaned quiz:
{
  "topic": "Literature",
  "questions": [
    {
      "question": "গল্পের থিম শনাক্ত করার প্রথম ধাপ কী?",
      "options": [
        "A) চরিত্রের বর্ণনা পড়া",
        "B) প্রশ্ন করা",
        "C) শব্দ গণনা করা",
        "D) ছবি আঁকা"
      ],
      "correct_answer": "B",
      "explanation": "প্রথম ধাপ হলো গল্পটি মনোযোগ দিয়ে পড়া এবং প্রশ্ন করা, যাতে থিমের ইঙ্গিত পাওয়া যায়।"
    },
    {
      "question": "থিমকে শক্তিশালী করার উপায় নয় কোনটি?",
      "options": [
        "A) প্রতীক ব্যবহার",
        "B) পুনরাবৃত্তি",
        "C) চরিত্রের নাম পরিবর্তন",
        "D) রূপক"
      ],
      "correct_answer": "C",
      "explanation": "চরিত্রের নাম পরিবর্তন থিমকে শক্তিশালী করে না; প্রতীক, পুনরাবৃত্তি ও রূপকই থিমকে জোরদার করে।"
    },
    {
      "question": "\"মেঘের ছায়া\" গল্পে কোন প্রতীক ব্যবহার করা হয়েছে এবং তা কী প্রকাশ করে?",
      "options": [
        "A) পাখি – স্বাধীনতা",
        "B) মেঘ – অস্থায়ী বাধা",
        "C) সূর্য – আশা",
        "D) নদী – প্রবাহ"
     

In [22]:
# ============================================================
# CELL 24 — PARSE QUIZ
# ============================================================

try:

    quiz_dict = json.loads(
        cleaned_quiz_text
    )

    print("✅ Valid JSON received.")

except json.JSONDecodeError as e:

    print("❌ Quiz JSON is invalid.")
    print("Error:", e)

    raise

✅ Valid JSON received.


In [23]:
# ============================================================
# CELL 25 — PYDANTIC VALIDATION
# ============================================================

try:

    quiz_data = Quiz.model_validate(
        quiz_dict
    )

    print("✅ Quiz successfully validated.")
    print(
        "Number of questions:",
        len(quiz_data.questions)
    )

except Exception as e:

    print("❌ Quiz validation failed.")
    print(type(e).__name__)
    print(str(e))

    raise

✅ Quiz successfully validated.
Number of questions: 5


In [24]:
# ============================================================
# CELL 26 — QUIZ VALIDATION
# ============================================================

if len(quiz_data.questions) != 5:

    raise ValueError(
        f"Expected 5 questions, "
        f"but received {len(quiz_data.questions)}."
    )


for i, question in enumerate(
    quiz_data.questions,
    start=1
):

    if len(question.options) != 4:

        raise ValueError(
            f"Question {i} does not contain "
            f"exactly 4 options."
        )


print("✅ Quiz structure is valid.")

✅ Quiz structure is valid.


In [25]:
# ============================================================
# CELL 27 — DISPLAY QUIZ
# ============================================================

print("\n")
print("=" * 80)
print("📝 LEO QUIZ")
print("=" * 80)

for i, question in enumerate(
    quiz_data.questions,
    start=1
):

    print(f"\nQuestion {i}")
    print(question.question)

    for option in question.options:
        print(option)

    print()



📝 LEO QUIZ

Question 1
গল্পের থিম শনাক্ত করার প্রথম ধাপ কী?
A) চরিত্রের বর্ণনা পড়া
B) প্রশ্ন করা
C) শব্দ গণনা করা
D) ছবি আঁকা


Question 2
থিমকে শক্তিশালী করার উপায় নয় কোনটি?
A) প্রতীক ব্যবহার
B) পুনরাবৃত্তি
C) চরিত্রের নাম পরিবর্তন
D) রূপক


Question 3
"মেঘের ছায়া" গল্পে কোন প্রতীক ব্যবহার করা হয়েছে এবং তা কী প্রকাশ করে?
A) পাখি – স্বাধীনতা
B) মেঘ – অস্থায়ী বাধা
C) সূর্য – আশা
D) নদী – প্রবাহ


Question 4
থিম শনাক্ত করার সময় কোন প্রশ্নটি সবচেয়ে গুরুত্বপূর্ণ?
A) গল্পের শিরোনাম কী?
B) লেখক কে?
C) গল্পের মূল বার্তা কী?
D) চরিত্রের বয়স কত?


Question 5
সাব-থিম কী?
A) প্রধান থিমকে সমর্থনকারী ছোট ভাবনা
B) গল্পের শিরোনাম
C) চরিত্রের নাম
D) প্রতীক



In [26]:
# ============================================================
# CELL 28 — STUDENT ANSWERS
# ============================================================

student_answers = []

print("=" * 80)
print("✍️ SUBMIT YOUR ANSWERS")
print("=" * 80)


for i in range(
    len(quiz_data.questions)
):

    while True:

        answer = input(
            f"Question {i + 1} "
            "(A/B/C/D): "
        ).strip().upper()

        if answer in ["A", "B", "C", "D"]:

            student_answers.append(
                answer
            )

            break

        print(
            "❌ Please enter A, B, C, or D."
        )


print("\n✅ Answers submitted.")

print(
    "Answers:",
    student_answers
)

✍️ SUBMIT YOUR ANSWERS
Question 1 (A/B/C/D): B
Question 2 (A/B/C/D): A
Question 3 (A/B/C/D): C
Question 4 (A/B/C/D): B
Question 5 (A/B/C/D): A

✅ Answers submitted.
Answers: ['B', 'A', 'C', 'B', 'A']


In [27]:
# ============================================================
# CELL 29 — CALCULATE SCORE
# ============================================================

correct_count = 0

answer_results = []


for i, question in enumerate(
    quiz_data.questions
):

    student_answer = student_answers[i]

    correct_answer = (
        question.correct_answer
    )

    is_correct = (
        student_answer == correct_answer
    )

    if is_correct:

        correct_count += 1


    answer_results.append({
        "question_number": i + 1,
        "student_answer": student_answer,
        "correct_answer": correct_answer,
        "is_correct": is_correct
    })


total_questions = len(
    quiz_data.questions
)

score_percentage = (
    correct_count / total_questions
) * 100


print("=" * 80)
print("📊 SCORE")
print("=" * 80)

print(
    f"Correct: {correct_count}/{total_questions}"
)

print(
    f"Score: {score_percentage:.2f}%"
)

📊 SCORE
Correct: 2/5
Score: 40.00%


In [28]:
# ============================================================
# CELL 30 — PREPARE EVALUATOR INPUT
# ============================================================

quiz_for_evaluator = (
    quiz_data.model_dump_json(
        indent=2,
        ensure_ascii=False
    )
)

answers_for_evaluator = json.dumps(
    answer_results,
    indent=2,
    ensure_ascii=False
)

print("✅ Evaluator input prepared.")

✅ Evaluator input prepared.


In [29]:
# ============================================================
# CELL 31 — EVALUATOR TASK
# ============================================================

evaluator_task = Task(
    description=f"""
Student: {student_name}
Topic: {topic}
Level: {level}
Language: {language}

Quiz:

{quiz_for_evaluator}

Answer results:

{answers_for_evaluator}

The Python program already calculated the score.

Score:
{correct_count}/{total_questions}
({score_percentage:.2f}%)

Provide concise learning feedback.

You must:

1. Mention the score.
2. Identify which questions were wrong.
3. Briefly explain the mistakes.
4. Give 2 or 3 study recommendations.

Do not recalculate the score.
Do not change the correct answers.
Do not create new questions.

Respond in {language}.
""",

    expected_output="""
    A concise evaluation report containing:
    - Score
    - Incorrect questions
    - Brief explanations
    - Study recommendations
    """,

    agent=evaluator
)

print("✅ Evaluator task created.")

✅ Evaluator task created.


In [30]:
# ============================================================
# CELL 32 — EVALUATION CREW
# ============================================================

evaluation_crew = Crew(
    agents=[evaluator],
    tasks=[evaluator_task],
    process=Process.sequential,
    verbose=True
)

print("✅ Evaluation crew created.")

✅ Evaluation crew created.


In [31]:
# ============================================================
# CELL 33 — RUN EVALUATOR
# ============================================================

print("\n")
print("=" * 80)
print("LEO — PHASE 2")
print("EVALUATOR")
print("=" * 80)


try:

    evaluation_result = await evaluation_crew.kickoff_async()

    print("\n")
    print("=" * 80)
    print("✅ EVALUATION COMPLETED")
    print("=" * 80)

except Exception as e:

    print("\n")
    print("=" * 80)
    print("❌ EVALUATION ERROR")
    print("=" * 80)

    print("Error type:", type(e).__name__)
    print("Error:", str(e))



LEO — PHASE 2
EVALUATOR


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 1cb8c9a9-f349-4068-a396-fb7f33293bf1                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  Student: Sneha                                                                                                 │
│  Topic: Literature                                                                                              │
│  Level: Intermediate                                                                                            │
│  Language: Bangla                                                                                               │
│                                                                                                                 │
│  Quiz:                                                                                                          │
│                                                                                                                 │
│  {                                                                                                              │
│    "topic": "Literature",                                                                                       │
│    "questions": [                                                                                               │
│      {                                                                                                          │
│        "question": "গল্পের থিম শনাক্ত করার প্রথম ধাপ কী?",                                                               │
│        "options": [                                                                                             │
│          "A) চরিত্রের বর্ণনা পড়া",                                                                                    │
│          "B) প্রশ্ন করা",                                                                                          │
│          "C) শব্দ গণনা করা",                                                                                       │
│          "D) ছবি আঁকা"                                                                                             │
│        ],                                                                                                       │
│        "correct_answer": "B",                                                                                   │
│        "explanation": "প্রথম ধাপ হলো গল্পটি মনোযোগ দিয়ে পড়া এবং প্রশ্ন করা, যাতে থিমের ইঙ্গিত পাওয়া যায়।"                              │
│      },                                                                                                         │
│      {                                                                                                          │
│        "question": "থিমকে শক্তিশালী করার উপায় নয় কোনটি?",                                                                 │
│        "options": [                                                                                             │
│          "A) প্রতীক ব্যবহার",                                                                                       │
│          "B) পুনরাবৃত্তি",                                                                                           │
│          "C) চরিত্রের নাম পরিবর্তন",                                                                                  │
│          "D) রূপক"                                                                                               │
│        ],                                                                                                       │
│        "correct_answer": "C",                                                              

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Evaluator                                                                                               │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Student: Sneha                                                                                                 │
│  Topic: Literature                                                                                              │
│  Level: Intermediate                                                                                            │
│  Language: Bangla                                                                                               │
│                                                                                                                 │
│  Quiz:                                                                                                          │
│                                                                                                                 │
│  {                                                                                                              │
│    "topic": "Literature",                                                                                       │
│    "questions": [                                                                                               │
│      {                                                                                                          │
│        "question": "গল্পের থিম শনাক্ত করার প্রথম ধাপ কী?",                                                               │
│        "options": [                                                                                             │
│          "A) চরিত্রের বর্ণনা পড়া",                                                                                    │
│          "B) প্রশ্ন করা",                                                                                          │
│          "C) শব্দ গণনা করা",                                                                                       │
│          "D) ছবি আঁকা"                                                                                             │
│        ],                                                                                                       │
│        "correct_answer": "B",                                                                                   │
│        "explanation": "প্রথম ধাপ হলো গল্পটি মনোযোগ দিয়ে পড়া এবং প্রশ্ন করা, যাতে থিমের ইঙ্গিত পাওয়া যায়।"                              │
│      },                                                                                                         │
│      {                                                                                                          │
│        "question": "থিমকে শক্তিশালী করার উপায় নয় কোনটি?",                                                                 │
│        "options": [                                                                                             │
│          "A) প্রতীক ব্যবহার",                                                                                       │
│          "B) পুনরাবৃত্তি",                                                                                           │
│          "C) চরিত্রের নাম পরিবর্তন",                                                                                  │
│          "D) রূপক"                                                                                               │
│        ],                                                                                  

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Evaluator                                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **স্কোর:** ২/৫ (৪০.০০%)                                                                                          │
│                                                                                                                 │
│  **ভুল উত্তর দেওয়া প্রশ্নগুলো:**                                                                                        │
│  1. **প্রশ্ন ২** – আপনি “A) প্রতীক ব্যবহার” নির্বাচন করেছেন, কিন্তু সঠিক উত্তর “C) চরিত্রের নাম পরিবর্তন”। চরিত্রের নাম পরিবর্তন থিমকে শক্তিশালী   │
│  করে না; প্রতীক, পুনরাবৃত্তি ও রূপকই থিমকে জোরদার করে।                                                                         │
│  2. **প্রশ্ন ৩** – আপনি “C) সূর্য – আশা” নির্বাচন করেছেন, কিন্তু সঠিক উত্তর “B) মেঘ – অস্থায়ী বাধা”। গল্পে মেঘকে প্রতীক হিসেবে ব্যবহার করা হয়েছে,   │
│  যা অস্থায়ী বাধা ও শেষে আলো দেখার ইঙ্গিত দেয়।                                                                                 │
│  3. **প্রশ্ন ৪** – আপনি “B) লেখক কে?” নির্বাচন করেছেন, কিন্তু সঠিক উত্তর “C) গল্পের মূল বার্তা কী?”। থিম শনাক্ত করার সময় “গল্পের মূল বার্তা     │
│  কী?” প্রশ্নটি সবচেয়ে গুরুত্বপূর্ণ।                                                                                        │
│                                                                                                                 │
│  **অধ্যয়ন সুপারিশ:**                                                                                                │
│  1. **থিম শনাক্ত করার ধাপগুলো পুনরায় পর্যালোচনা করুন** – প্রথমে গল্প মনোযোগ দিয়ে পড়া, তারপর মূল প্রশ্ন করা।                                │
│  2. **প্রতীক ও রূপকের উদাহরণগুলো অনুশীলন করুন** – বিভিন্ন গল্পে কোন বস্তু বা চিত্র কীভাবে থিমকে প্রকাশ করে তা চিহ্নিত করার চেষ্টা করুন।               │
│  3. **প্রধান প্রশ্নগুলোর তালিকা তৈরি করুন** – থিম নির্ণয়ের সময় কোন প্রশ্নগুলো সবচেয়ে কার্যকর তা মনে রাখার জন্য একটি ছোট চেকলিস্ট বানিয়ে নিন।           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Student: Sneha                                                                                                 │
│  Topic: Literature                                                                                              │
│  Level: Intermediate                                                                                            │
│  Language: Bangla                                                                                               │
│                                                                                                                 │
│  Quiz:                                                                                                          │
│                                                                                                                 │
│  {                                                                                                              │
│    "topic": "Literature",                                                                                       │
│    "questions": [                                                                                               │
│      {                                                                                                          │
│        "question": "গল্পের থিম শনাক্ত করার প্রথম ধাপ কী?",                                                               │
│        "options": [                                                                                             │
│          "A) চরিত্রের বর্ণনা পড়া",                                                                                    │
│          "B) প্রশ্ন করা",                                                                                          │
│          "C) শব্দ গণনা করা",                                                                                       │
│          "D) ছবি আঁকা"                                                                                             │
│        ],                                                                                                       │
│        "correct_answer": "B",                                                                                   │
│        "explanation": "প্রথম ধাপ হলো গল্পটি মনোযোগ দিয়ে পড়া এবং প্রশ্ন করা, যাতে থিমের ইঙ্গিত পাওয়া যায়।"                              │
│      },                                                                                                         │
│      {                                                                                                          │
│        "question": "থিমকে শক্তিশালী করার উপায় নয় কোনটি?",                                                                 │
│        "options": [                                                                                             │
│          "A) প্রতীক ব্যবহার",                                                                                       │
│          "B) পুনরাবৃত্তি",                                                                                           │
│          "C) চরিত্রের নাম পরিবর্তন",                                                                                  │
│          "D) রূপক"                                                                                               │
│        ],                                                                                                       │
│        "correct_answer": "C",                                                              



✅ EVALUATION COMPLETED


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 1cb8c9a9-f349-4068-a396-fb7f33293bf1                                                                       │
│  Final Output: **স্কোর:** ২/৫ (৪০.০০%)                                                                            │
│                                                                                                                 │
│  **ভুল উত্তর দেওয়া প্রশ্নগুলো:**                                                                                        │
│  1. **প্রশ্ন ২** – আপনি “A) প্রতীক ব্যবহার” নির্বাচন করেছেন, কিন্তু সঠিক উত্তর “C) চরিত্রের নাম পরিবর্তন”। চরিত্রের নাম পরিবর্তন থিমকে শক্তিশালী   │
│  করে না; প্রতীক, পুনরাবৃত্তি ও রূপকই থিমকে জোরদার করে।                                                                         │
│  2. **প্রশ্ন ৩** – আপনি “C) সূর্য – আশা” নির্বাচন করেছেন, কিন্তু সঠিক উত্তর “B) মেঘ – অস্থায়ী বাধা”। গল্পে মেঘকে প্রতীক হিসেবে ব্যবহার করা হয়েছে,   │
│  যা অস্থায়ী বাধা ও শেষে আলো দেখার ইঙ্গিত দেয়।                                                                                 │
│  3. **প্রশ্ন ৪** – আপনি “B) লেখক কে?” নির্বাচন করেছেন, কিন্তু সঠিক উত্তর “C) গল্পের মূল বার্তা কী?”। থিম শনাক্ত করার সময় “গল্পের মূল বার্তা     │
│  কী?” প্রশ্নটি সবচেয়ে গুরুত্বপূর্ণ।                                                                                        │
│                                                                                                                 │
│  **অধ্যয়ন সুপারিশ:**                                                                                                │
│  1. **থিম শনাক্ত করার ধাপগুলো পুনরায় পর্যালোচনা করুন** – প্রথমে গল্প মনোযোগ দিয়ে পড়া, তারপর মূল প্রশ্ন করা।                                │
│  2. **প্রতীক ও রূপকের উদাহরণগুলো অনুশীলন করুন** – বিভিন্ন গল্পে কোন বস্তু বা চিত্র কীভাবে থিমকে প্রকাশ করে তা চিহ্নিত করার চেষ্টা করুন।               │
│  3. **প্রধান প্রশ্নগুলোর তালিকা তৈরি করুন** – থিম নির্ণয়ের সময় কোন প্রশ্নগুলো সবচেয়ে কার্যকর তা মনে রাখার জন্য একটি ছোট চেকলিস্ট বানিয়ে নিন।           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [32]:
# ============================================================
# CELL 34 — FINAL REPORT
# ============================================================

print("\n")
print("=" * 80)
print("🎓 LEO FINAL LEARNING REPORT")
print("=" * 80)

print(f"Student : {student_name}")
print(f"Topic   : {topic}")
print(f"Level   : {level}")
print(f"Language: {language}")

print("\nQuiz Result")
print("-" * 80)

print(
    f"Correct answers: "
    f"{correct_count}/{total_questions}"
)

print(
    f"Score: "
    f"{score_percentage:.2f}%"
)

print("\nEvaluator Feedback")
print("-" * 80)

print(
    evaluation_result.raw
)

print("\n")
print("=" * 80)
print("🎉 LEO SESSION COMPLETED")
print("=" * 80)



🎓 LEO FINAL LEARNING REPORT
Student : Sneha
Topic   : Literature
Level   : Intermediate
Language: Bangla

Quiz Result
--------------------------------------------------------------------------------
Correct answers: 2/5
Score: 40.00%

Evaluator Feedback
--------------------------------------------------------------------------------
**স্কোর:** ২/৫ (৪০.০০%)

**ভুল উত্তর দেওয়া প্রশ্নগুলো:**
1. **প্রশ্ন ২** – আপনি “A) প্রতীক ব্যবহার” নির্বাচন করেছেন, কিন্তু সঠিক উত্তর “C) চরিত্রের নাম পরিবর্তন”। চরিত্রের নাম পরিবর্তন থিমকে শক্তিশালী করে না; প্রতীক, পুনরাবৃত্তি ও রূপকই থিমকে জোরদার করে।  
2. **প্রশ্ন ৩** – আপনি “C) সূর্য – আশা” নির্বাচন করেছেন, কিন্তু সঠিক উত্তর “B) মেঘ – অস্থায়ী বাধা”। গল্পে মেঘকে প্রতীক হিসেবে ব্যবহার করা হয়েছে, যা অস্থায়ী বাধা ও শেষে আলো দেখার ইঙ্গিত দেয়।  
3. **প্রশ্ন ৪** – আপনি “B) লেখক কে?” নির্বাচন করেছেন, কিন্তু সঠিক উত্তর “C) গল্পের মূল বার্তা কী?”। থিম শনাক্ত করার সময় “গল্পের মূল বার্তা কী?” প্রশ্নটি সবচেয়ে গুরুত্বপূর্ণ।

**অধ্যয়ন সুপারিশ:**
1. **থিম শনা

In [33]:
# ============================================================
# CELL 35 — UPDATE MEMORY
# ============================================================

leo_memory["quiz_score"] = score_percentage

print("=" * 60)
print("🧠 LEO MEMORY")
print("=" * 60)

for key, value in leo_memory.items():

    print(
        f"{key}: {value}"
    )

🧠 LEO MEMORY
student_name: Sneha
topic: Literature
level: Intermediate
language: Bangla
quiz_score: 40.0


In [34]:
# ============================================================
# CELL 38 — SAVE SESSION SUMMARY
# ============================================================

session_summary = {
    "student_name": student_name,
    "topic": topic,
    "level": level,
    "language": language,
    "total_questions": total_questions,
    "correct_answers": correct_count,
    "score_percentage": round(
        score_percentage,
        2
    )
}

with open(
    "leo_session_summary.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        session_summary,
        f,
        indent=4,
        ensure_ascii=False
    )


print("✅ Session summary saved.")

print(
    json.dumps(
        session_summary,
        indent=4,
        ensure_ascii=False
    )
)

✅ Session summary saved.
{
    "student_name": "Sneha",
    "topic": "Literature",
    "level": "Intermediate",
    "language": "Bangla",
    "total_questions": 5,
    "correct_answers": 2,
    "score_percentage": 40.0
}


In [35]:
!pip install -q gradio

In [36]:
# ============================================================
# CELL 40 — GRADIO INTERFACE FOR LEO
# ============================================================

import gradio as gr
import json
import re


# ------------------------------------------------------------
# Helper Function
# ------------------------------------------------------------

def clean_json_text(text):
    text = text.strip()

    text = re.sub(
        r"^```json\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\s*```$",
        "",
        text
    )

    return text.strip()


# ============================================================
# MAIN LEARNING FUNCTION
# ============================================================

async def start_learning(
    student_name,
    topic,
    level,
    language
):

    # --------------------------------------------------------
    # Validate Input
    # --------------------------------------------------------

    if not student_name.strip():
        return (
            "❌ Please enter your name.",
            "",
            "",
            None
        )

    if not topic.strip():
        return (
            "❌ Please enter a topic.",
            "",
            "",
            None
        )

    # --------------------------------------------------------
    # Update Leo Memory
    # --------------------------------------------------------

    leo_memory["student_name"] = student_name
    leo_memory["topic"] = topic
    leo_memory["level"] = level
    leo_memory["language"] = language

    inputs = {
        "student_name": student_name,
        "topic": topic,
        "level": level,
        "language": language
    }

    # --------------------------------------------------------
    # COORDINATOR
    # --------------------------------------------------------

    coordinator_status = """
### 🔄 Coordinator Agent is working...

The Coordinator is analyzing the student's request
and organizing the learning workflow.
"""

    try:

        coordinator_ui_task = Task(
            description=f"""
Student: {student_name}
Topic: {topic}
Level: {level}
Language: {language}

Analyze the student's learning request.

Create a concise workflow containing:
1. Topic explanation
2. Practice quiz
3. Evaluation and feedback

Focus only on organizing the workflow.

Do not teach the topic.
Do not create quiz questions.
""",
            expected_output="""
A concise learning workflow.
""",
            agent=coordinator
        )

        coordinator_crew = Crew(
            agents=[coordinator],
            tasks=[coordinator_ui_task],
            process=Process.sequential,
            verbose=False
        )

        coordinator_result = await coordinator_crew.kickoff_async(
            inputs=inputs
        )

        coordinator_output = coordinator_result.raw

    except Exception as e:

        return (
            f"{coordinator_status}\n\n"
            f"❌ **Coordinator Error**\n\n"
            f"`{str(e)}`",
            "",
            "",
            None
        )

    # --------------------------------------------------------
    # EXPLAINER
    # --------------------------------------------------------

    explainer_status = """
### ✅ Coordinator completed

### 🔄 Explainer Agent is working...

The Explainer is preparing the lesson.
"""

    try:

        explainer_ui_task = Task(
            description=f"""
Student: {student_name}
Topic: {topic}
Level: {level}
Language: {language}

Coordinator workflow:

{coordinator_output}

Teach the requested topic.

Requirements:
- Adapt the explanation to the student's level.
- Use the requested language.
- Explain important concepts clearly.
- Include simple examples.
- Prepare the student for the quiz.

Do not create quiz questions.
""",
            expected_output="""
A clear lesson suitable for the student's level.
""",
            agent=explainer
        )

        explainer_crew = Crew(
            agents=[explainer],
            tasks=[explainer_ui_task],
            process=Process.sequential,
            verbose=False
        )

        explainer_result = await explainer_crew.kickoff_async(
            inputs=inputs
        )

        lesson_text = explainer_result.raw

    except Exception as e:

        return (
            f"{explainer_status}\n\n"
            f"❌ **Explainer Error**\n\n"
            f"`{str(e)}`",
            "",
            "",
            None
        )

    # --------------------------------------------------------
    # QUIZ MASTER
    # --------------------------------------------------------

    quiz_status = """
### ✅ Coordinator completed

### ✅ Explainer completed

### 🔄 Quiz Master Agent is working...

The Quiz Master is creating 5 MCQs based on the lesson.
"""

    try:

        quiz_ui_task = Task(
            description=f"""
Student: {student_name}
Topic: {topic}
Level: {level}
Language: {language}

Use this lesson:

{lesson_text}

Create exactly 5 multiple-choice questions.

Each question must contain:
- question
- options
- correct_answer
- explanation

Each question must have exactly four options:

A) ...
B) ...
C) ...
D) ...

correct_answer must be exactly:
A, B, C, or D

Return ONLY valid JSON.

Use this structure:

{{
  "topic": "{topic}",
  "questions": [
    {{
      "question": "Question text",
      "options": [
        "A) Option",
        "B) Option",
        "C) Option",
        "D) Option"
      ],
      "correct_answer": "A",
      "explanation": "Explanation"
    }}
  ]
}}

Do not use Markdown.
Do not use ```json.
Do not add text before or after the JSON.
""",
            expected_output="""
Valid JSON containing exactly 5 MCQs.
""",
            agent=quiz_master
        )

        quiz_crew = Crew(
            agents=[quiz_master],
            tasks=[quiz_ui_task],
            process=Process.sequential,
            verbose=False
        )

        quiz_result = await quiz_crew.kickoff_async(
            inputs=inputs
        )

        quiz_text = clean_json_text(
            quiz_result.raw
        )

        quiz_dict = json.loads(
            quiz_text
        )

        quiz_data = Quiz.model_validate(
            quiz_dict
        )

        # Validate number of questions
        if len(quiz_data.questions) != 5:
            raise ValueError(
                "Quiz must contain exactly 5 questions."
            )

        # Validate options
        for question in quiz_data.questions:

            if len(question.options) != 4:
                raise ValueError(
                    "Every question must have exactly 4 options."
                )

    except Exception as e:

        return (
            f"{quiz_status}\n\n"
            f"❌ **Quiz Master Error**\n\n"
            f"`{str(e)}`",
            lesson_text,
            "",
            None
        )

    # --------------------------------------------------------
    # FORMAT QUIZ
    # --------------------------------------------------------

    quiz_display = ""

    for i, question in enumerate(
        quiz_data.questions,
        start=1
    ):

        quiz_display += (
            f"### Question {i}\n\n"
        )

        quiz_display += (
            f"**{question.question}**\n\n"
        )

        for option in question.options:

            quiz_display += (
                f"- {option}\n"
            )

        quiz_display += "\n"

    final_status = """
### ✅ Coordinator completed

### ✅ Explainer completed

### ✅ Quiz Master completed

### 📝 Quiz Ready

Answer all 5 questions and click **Submit Quiz**.
"""

    quiz_json = quiz_data.model_dump_json()

    return (
        final_status,
        lesson_text,
        quiz_display,
        quiz_json
    )


# ============================================================
# SUBMIT QUIZ
# ============================================================

async def submit_quiz(
    student_name,
    topic,
    level,
    language,
    answer1,
    answer2,
    answer3,
    answer4,
    answer5,
    quiz_json
):

    if not quiz_json:

        return (
            "❌ Please click **Start Learning** first.",
            ""
        )

    # --------------------------------------------------------
    # Load Quiz
    # --------------------------------------------------------

    try:

        quiz_dict = json.loads(
            quiz_json
        )

        quiz_data = Quiz.model_validate(
            quiz_dict
        )

    except Exception as e:

        return (
            f"❌ Quiz data error:\n\n`{str(e)}`",
            ""
        )

    # --------------------------------------------------------
    # Student Answers
    # --------------------------------------------------------

    student_answers = [
        answer1,
        answer2,
        answer3,
        answer4,
        answer5
    ]

    # --------------------------------------------------------
    # Validate Answers
    # --------------------------------------------------------

    for i, answer in enumerate(
        student_answers,
        start=1
    ):

        if answer not in ["A", "B", "C", "D"]:

            return (
                f"❌ Please answer Question {i}.",
                ""
            )

    # --------------------------------------------------------
    # PYTHON SCORE CALCULATION
    # --------------------------------------------------------

    correct_count = 0

    answer_results = []

    for i, question in enumerate(
        quiz_data.questions
    ):

        student_answer = student_answers[i]

        correct_answer = question.correct_answer

        is_correct = (
            student_answer == correct_answer
        )

        if is_correct:
            correct_count += 1

        answer_results.append({
            "question_number": i + 1,
            "student_answer": student_answer,
            "correct_answer": correct_answer,
            "is_correct": is_correct
        })

    total_questions = len(
        quiz_data.questions
    )

    score_percentage = (
        correct_count /
        total_questions
    ) * 100

    # --------------------------------------------------------
    # EVALUATOR
    # --------------------------------------------------------

    evaluator_status = f"""
### 🧮 Python Score Calculation Completed

**Score: {correct_count}/{total_questions}**
**Percentage: {score_percentage:.0f}%**

### 🔄 Evaluator Agent is working...

The Evaluator is preparing personalized feedback.
"""

    quiz_for_evaluator = quiz_data.model_dump_json(
        indent=2
    )

    answers_for_evaluator = json.dumps(
        answer_results,
        indent=2
    )

    try:

        evaluator_ui_task = Task(
            description=f"""
Student: {student_name}
Topic: {topic}
Level: {level}
Language: {language}

Python has already calculated the score.

Score:
{correct_count}/{total_questions}

Percentage:
{score_percentage:.0f}%

Quiz:
{quiz_for_evaluator}

Student Answer Results:
{answers_for_evaluator}

Provide concise personalized feedback.

Include:
1. Overall performance
2. Incorrect questions
3. Brief explanation of mistakes
4. 2–3 recommendations

Do not recalculate the score.
Do not change the correct answers.

Respond in:
{language}
""",
            expected_output="""
Concise personalized feedback.
""",
            agent=evaluator
        )

        evaluator_crew = Crew(
            agents=[evaluator],
            tasks=[evaluator_ui_task],
            process=Process.sequential,
            verbose=False
        )

        evaluator_result = await evaluator_crew.kickoff_async()

        feedback = evaluator_result.raw

    except Exception as e:

        return (
            f"{evaluator_status}\n\n"
            f"❌ **Evaluator Error**\n\n"
            f"`{str(e)}`",
            ""
        )

    # --------------------------------------------------------
    # UPDATE MEMORY
    # --------------------------------------------------------

    leo_memory["quiz_score"] = score_percentage

    # --------------------------------------------------------
    # FINAL RESULT
    # --------------------------------------------------------

    final_status = f"""
### ✅ Evaluation Completed

**Student:** {student_name}

**Topic:** {topic}

**Level:** {level}

**Score:** {correct_count}/{total_questions}

**Percentage:** {score_percentage:.0f}%

---

### 🤖 Agent Workflow

✅ Coordinator
✅ Explainer
✅ Quiz Master
✅ Python Score Calculation
✅ Evaluator
"""

    return (
        final_status,
        feedback
    )


# ============================================================
# GRADIO INTERFACE
# ============================================================

with gr.Blocks(
    title="Leo — Multi-Agent AI Tutor"
) as demo:

    gr.Markdown(
        """
        # 🎓 Leo — Multi-Agent AI Tutor

        ### Learn → Practice → Evaluate

        **Coordinator → Explainer → Quiz Master → Evaluator**
        """
    )

    # --------------------------------------------------------
    # STUDENT INFORMATION
    # --------------------------------------------------------

    gr.Markdown(
        "## 👤 Student Information"
    )

    with gr.Row():

        student_name = gr.Textbox(
            label="Student Name",
            placeholder="Enter your name"
        )

        topic = gr.Textbox(
            label="Topic",
            placeholder="e.g. Literature, Python, Machine Learning"
        )

    with gr.Row():

        level = gr.Dropdown(
            choices=[
                "Beginner",
                "Intermediate",
                "Advanced"
            ],
            value="Beginner",
            label="Learning Level"
        )

        language = gr.Dropdown(
            choices=[
                "English",
                "Bangla"
            ],
            value="English",
            label="Preferred Language"
        )

    start_button = gr.Button(
        "🚀 Start Learning",
        variant="primary"
    )

    # --------------------------------------------------------
    # STATUS
    # --------------------------------------------------------

    status_output = gr.Markdown(
        value="Waiting for student input..."
    )

    # --------------------------------------------------------
    # LESSON
    # --------------------------------------------------------

    with gr.Tab("📚 Lesson"):

        lesson_output = gr.Markdown(
            value="Lesson will appear here."
        )

    # --------------------------------------------------------
    # QUIZ
    # --------------------------------------------------------

    with gr.Tab("📝 Quiz"):

        quiz_output = gr.Markdown(
            value="Quiz will appear here."
        )

        gr.Markdown(
            "### Select Your Answers"
        )

        with gr.Row():

            answer1 = gr.Radio(
                ["A", "B", "C", "D"],
                label="Question 1"
            )

            answer2 = gr.Radio(
                ["A", "B", "C", "D"],
                label="Question 2"
            )

        with gr.Row():

            answer3 = gr.Radio(
                ["A", "B", "C", "D"],
                label="Question 3"
            )

            answer4 = gr.Radio(
                ["A", "B", "C", "D"],
                label="Question 4"
            )

        answer5 = gr.Radio(
            ["A", "B", "C", "D"],
            label="Question 5"
        )

        submit_button = gr.Button(
            "📊 Submit Quiz",
            variant="primary"
        )

    # --------------------------------------------------------
    # EVALUATION
    # --------------------------------------------------------

    with gr.Tab("📊 Evaluation"):

        evaluation_output = gr.Markdown(
            value="Evaluation will appear here."
        )

    # --------------------------------------------------------
    # HIDDEN QUIZ STATE
    # --------------------------------------------------------

    quiz_state = gr.State(
        value=None
    )

    # --------------------------------------------------------
    # START LEARNING
    # --------------------------------------------------------

    start_button.click(
        fn=start_learning,
        inputs=[
            student_name,
            topic,
            level,
            language
        ],
        outputs=[
            status_output,
            lesson_output,
            quiz_output,
            quiz_state
        ]
    )

    # --------------------------------------------------------
    # SUBMIT QUIZ
    # --------------------------------------------------------

    submit_button.click(
        fn=submit_quiz,
        inputs=[
            student_name,
            topic,
            level,
            language,
            answer1,
            answer2,
            answer3,
            answer4,
            answer5,
            quiz_state
        ],
        outputs=[
            status_output,
            evaluation_output
        ]
    )


# ============================================================
# LAUNCH
# ============================================================

demo.launch(
    share=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5f41520317a2958c35.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
